# 🤖 Classification — E1–E7

Dự đoán `has_recorded_change` (khối code có ghi nhận lần thay đổi đầu tiên hay không) từ kích thước code, repo, ngôn ngữ, công cụ AI và thời gian. Đây là bài toán dự đoán, không phải chứng minh nguyên nhân.

Train/test và cross-validation đều chia theo repo, để các khối của cùng một repo không nằm ở cả hai tập. Chia ngẫu nhiên theo dòng làm AUC cao giả tạo vì mô hình nhận ra các khối lặp lại trong cùng repo.

Cần cài thêm: `scikit-learn`.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, RocCurveDisplay
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
DATA_PATH = Path('../data/processed/commits_features.csv')
REPORT_DIR = Path('../reports'); FIG_DIR = REPORT_DIR / 'figures'; FIG_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_PATH, low_memory=False).replace([np.inf,-np.inf],np.nan)
feature_cols = ['code_lines','code_chars','repo_commit_count','language','ai_tool','ai_verb','year','hour','is_weekend']
df = df.dropna(subset=feature_cols + ['has_recorded_change']).copy()
X = df[feature_cols]; y = df['has_recorded_change'].astype(int); groups = df['repo']
numeric_features = ['code_lines','code_chars','repo_commit_count','year','hour']
categorical_features = ['language','ai_tool','ai_verb','is_weekend']
preprocessor = ColumnTransformer([('num', StandardScaler(), numeric_features), ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])
models = {'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced'), 'Decision Tree': DecisionTreeClassifier(max_depth=12, min_samples_leaf=10, class_weight='balanced', random_state=42), 'Random Forest': RandomForestClassifier(n_estimators=250, max_depth=16, min_samples_leaf=3, class_weight='balanced', n_jobs=-1, random_state=42)}
# Chia theo repo: khối của cùng một repo không nằm ở cả train và test (tránh rò rỉ dữ liệu)
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=.20, random_state=42).split(X, y, groups))
X_train, X_test, y_train, y_test = X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]
print(f'Dataset: {len(df):,} khối, {groups.nunique():,} repo; train={len(X_train):,}; test={len(X_test):,} ({groups.iloc[test_idx].nunique():,} repo); positive rate={y.mean():.3f}')

Dataset: 21,498 khối, 12,249 repo; train=17,712; test=3,786 (2,450 repo); positive rate=0.349


## E1–E4 — Train/test và huấn luyện 3 mô hình

In [2]:
pipelines = {name: Pipeline([('preprocess', preprocessor), ('model', model)]) for name, model in models.items()}
fitted = {}
predictions = {}
probabilities = {}
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train); fitted[name] = pipe
    predictions[name] = pipe.predict(X_test); probabilities[name] = pipe.predict_proba(X_test)[:,1]
    print(f'--- {name} ---'); print(classification_report(y_test, predictions[name], target_names=['No recorded change','Recorded change']))

--- Logistic Regression ---
                    precision    recall  f1-score   support

No recorded change       0.72      0.82      0.76      2623
   Recorded change       0.41      0.29      0.34      1163

          accuracy                           0.65      3786
         macro avg       0.56      0.55      0.55      3786
      weighted avg       0.62      0.65      0.63      3786

--- Decision Tree ---
                    precision    recall  f1-score   support

No recorded change       0.72      0.58      0.64      2623
   Recorded change       0.34      0.50      0.41      1163

          accuracy                           0.55      3786
         macro avg       0.53      0.54      0.53      3786
      weighted avg       0.61      0.55      0.57      3786



--- Random Forest ---
                    precision    recall  f1-score   support

No recorded change       0.74      0.71      0.72      2623
   Recorded change       0.40      0.44      0.42      1163

          accuracy                           0.63      3786
         macro avg       0.57      0.57      0.57      3786
      weighted avg       0.64      0.63      0.63      3786



## E5 — Accuracy, Precision, Recall, F1, ROC-AUC và confusion matrix

In [3]:
metric_rows = []
for name in models:
    pred, prob = predictions[name], probabilities[name]
    metric_rows.append({'model':name, 'accuracy':accuracy_score(y_test,pred), 'precision':precision_score(y_test,pred), 'recall':recall_score(y_test,pred), 'f1':f1_score(y_test,pred), 'roc_auc':roc_auc_score(y_test,prob)})
metrics = pd.DataFrame(metric_rows).sort_values('roc_auc', ascending=False)
display(metrics); metrics.to_csv(REPORT_DIR / 'classification_metrics.csv', index=False)
fig, axes = plt.subplots(1,3,figsize=(15,4))
for ax,(name,pred) in zip(axes,predictions.items()): ConfusionMatrixDisplay(confusion_matrix(y_test,pred), display_labels=['No recorded change','Recorded change']).plot(ax=ax, colorbar=False); ax.set_title(name)
fig.tight_layout(); fig.savefig(FIG_DIR / 'E5_confusion_matrices.png', dpi=160, bbox_inches='tight'); plt.show(); plt.close(fig)
fig, ax = plt.subplots(figsize=(7,5))
for name,prob in probabilities.items(): RocCurveDisplay.from_predictions(y_test,prob,name=name,ax=ax)
ax.set_title('ROC curves'); fig.tight_layout(); fig.savefig(FIG_DIR / 'E5_roc_curves.png', dpi=160, bbox_inches='tight'); plt.show(); plt.close(fig)

,model,accuracy,precision,recall,f1,roc_auc
2,Random Forest,0.626255,0.400631,0.436801,0.417935,0.609449
0,Logistic Regression,0.652404,0.406365,0.285469,0.335354,0.567980
1,Decision Tree,0.554411,0.344603,0.499570,0.407862,0.555382


## E6 — Feature importance

In [4]:
importance_rows = []
for name, pipe in fitted.items():
    model = pipe.named_steps['model']; prep = pipe.named_steps['preprocess']; names = prep.get_feature_names_out()
    values = model.coef_[0] if hasattr(model,'coef_') else model.feature_importances_
    for feature, value in zip(names, values): importance_rows.append({'model':name, 'feature':feature, 'importance':value, 'abs_importance':abs(value)})
importance = pd.DataFrame(importance_rows)
display(importance.sort_values(['model','abs_importance'], ascending=[True,False]).groupby('model').head(15))
importance.to_csv(REPORT_DIR / 'classification_feature_importance.csv', index=False)
rf_top = importance[importance.model.eq('Random Forest')].nlargest(15,'abs_importance').sort_values('importance')
plt.figure(figsize=(9,6)); sns.barplot(data=rf_top,x='importance',y='feature',color='#16a34a'); plt.title('Top 15 Random Forest feature importance'); plt.xlabel('Importance'); plt.ylabel('Feature'); plt.tight_layout(); plt.savefig(FIG_DIR / 'E6_feature_importance.png',dpi=160,bbox_inches='tight'); plt.show(); plt.close()

,model,feature,importance,abs_importance
19,Decision Tree,num__code_chars,0.304292,0.304292
20,Decision Tree,num__repo_commit_count,0.218344,0.218344
22,Decision Tree,num__hour,0.143574,0.143574
18,Decision Tree,num__code_lines,0.131968,0.131968
21,Decision Tree,num__year,0.067866,0.067866
30,Decision Tree,cat__ai_tool_other_ai,0.028333,0.028333
25,Decision Tree,cat__ai_tool_chatgpt_gpt,0.027474,0.027474
24,Decision Tree,cat__language_python,0.019253,0.019253
23,Decision Tree,cat__language_js,0.012312,0.012312
28,Decision Tree,cat__ai_tool_github_copilot,0.010032,0.010032


## E7 — 5-fold GroupKFold theo repo

In [5]:
cv = GroupKFold(n_splits=5)
cv_rows = []
scoring = {'accuracy':'accuracy','precision':'precision','recall':'recall','f1':'f1','roc_auc':'roc_auc'}
for name, pipe in pipelines.items():
    result = cross_validate(pipe, X, y, groups=groups, cv=cv, scoring=scoring, n_jobs=-1)
    row = {'model':name}
    for metric in scoring: row[f'{metric}_mean'] = result[f'test_{metric}'].mean(); row[f'{metric}_std'] = result[f'test_{metric}'].std()
    cv_rows.append(row)
cv_table = pd.DataFrame(cv_rows).sort_values('roc_auc_mean', ascending=False)
display(cv_table); cv_table.to_csv(REPORT_DIR / 'classification_cross_validation.csv', index=False)
print('✅ Đã lưu metrics, feature importance, confusion matrix, ROC curve và cross-validation vào reports/.')

,model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
2,Random Forest,0.637035,0.012480,0.471241,0.049992,0.505965,0.084155,0.487584,0.065519,0.652957,0.016673
1,Decision Tree,0.593682,0.014548,0.428545,0.055704,0.567319,0.097654,0.487541,0.071114,0.617030,0.028850
0,Logistic Regression,0.632572,0.017807,0.446253,0.054681,0.378379,0.171051,0.400106,0.109596,0.596195,0.037779


✅ Đã lưu metrics, feature importance, confusion matrix, ROC curve và cross-validation vào reports/.
